# Fehlende Radwege aus Mapillary-Verkehrszeichen → MapRoulette

Schlankere Fassung von `1_merge_mapillary-trafficsigns_osm-cycleways.ipynb`.
Die Logik liegt in [`cw_campaign.py`](cw_campaign.py), dieses Notebook ruft sie nur auf
und zeigt die Zwischenstände. Tests: `pytest test_cw_campaign.py`.

**Ablauf**

| Schritt | Funktion |
| --- | --- |
| Verkehrszeichen von data.vizsim.de spiegeln | `sync_features` |
| radverkehrsbezogene Zeichen einlesen | `load_features` |
| auf Deutschland beschneiden | `clip_to_boundary` |
| frische / kurzlebige Schilder aussortieren | `filter_stable_signs` |
| Abstand zur nächsten OSM-Radinfra und Autobahn | `distance_to_nearest` |
| Priorität aus dem Abstand ableiten | `assign_priority` |
| bereits bestehende Aufgaben abziehen | `drop_near_existing_tasks` |
| **neuestes** Mapillary-Bild je Zeichen holen | `newest_image_ids` |
| GeoJSON für MapRoulette schreiben | `build_maproulette_geojson` |

**Was gegenüber `1_` anders ist** — ausführlich in [`1b_unterschiede.md`](1b_unterschiede.md):

1. **Das verlinkte Foto ist jetzt das neueste.** `1_` nahm das letzte Element einer
   unsortierten Liste. An 600 Bremer Radwegzeichen war das in 84 % der Fälle nicht das
   neueste Bild, in 21 % lag über ein Monat dazwischen, in 10 % über ein Jahr.
2. **Eine Abstandsspalte statt zweier Pufferläufe.** `dist_cw_m` ersetzt
   `df_buffered_25` / `df_buffered_30`; die Schwellen 25 m und 30 m stehen in
   `PRIO_AB_DISTANZ`. Die Aufgabentexte nennen dadurch den echten Abstand.
3. **Sammelabfragen statt Einzelabrufe** (50 Features pro Request) — an 200 Features
   1,9 s statt 30,5 s. Die `mapillary`-Library wird nicht mehr gebraucht.
4. **`sidewalk:bicycle` zählt als Radinfra** (das TODO in `1_`, Zelle 5).

Geschrieben wird in dieselbe Datei wie bisher
(`maproulette_tasks_missing-cw_instruction_vz_name_new.geojson`), damit der Eingabepfad in
MapRoulette gleich bleibt. Der letzte Abschnitt stellt den vorherigen Stand gegenüber.

## Einstellungen

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# cw_campaign.py liegt eine Ebene höher, weil beide Cycleway-Kampagnen es nutzen.
# Das ".." funktioniert, weil Jupyter und nbconvert das Arbeitsverzeichnis auf das
# Notebook-Verzeichnis setzen — dieselbe Annahme wie bei ../../output/ und ../utils/.
import sys

sys.path.insert(0, "..")
import cw_campaign as cw

# Muss zum Datum in ../0_prepare_osm_network.ipynb passen - das Notebook
# erzeugt die Radwege-Datei für beide Cycleway-Kampagnen.
set_date = "260915"

ordner_zeichen = Path("../../output")
pfad_radwege = Path(f"../utils/processed_osm_files/processed_cycleways_germany_{set_date}.parquet")
pfad_autobahnen = Path("../utils/processed_motorways_germany_251215.parquet")  # ändert sich kaum
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
pfad_config = Path("../utils/config_mapillary_privat.json")

# Nur Schilder, die nach diesem Datum zuletzt gesehen wurden ...
zuletzt_gesehen_nach = "2025-07-01"
# ... und die so viele Monate zwischen erster und letzter Sichtung liegen haben.
# Siebt Baustellen- und Umleitungsbeschilderung aus.
mindest_standzeit_monate = 9

challenge_id = 52916

# Fester Name: MapRoulette zieht die Aufgaben immer aus demselben Pfad.
# Der vorherige Stand steckt in der git-History, falls mal etwas zurückmuss.
ziel_geojson = Path("maproulette_tasks_missing-cw_instruction_vz_name_new.geojson")

print("Prioritätsschwellen:", cw.PRIO_AB_DISTANZ)
print("Zeichen:", ", ".join(f"DE:{v}" for v in sorted(set(cw.RADWEG_ZEICHEN.values()))))

## 1 · Verkehrszeichen holen und einlesen

In [ ]:
# Lädt nur, was lokal fehlt oder auf dem Server neuer ist.
metadata = cw.sync_features(ordner_zeichen)
print("Datenstand:", metadata["processed_date"])

In [ ]:
signs = cw.load_features(ordner_zeichen, prefix=cw.PREFIX_ZEICHEN, values=cw.RADWEG_ZEICHEN)
signs.head()

In [ ]:
# Die Parquets sind nach Zoom-14-Kacheln geschnitten, Randkacheln ragen ins Ausland.
signs = cw.clip_to_boundary(signs, cw.load_boundary(pfad_grenze))

## 2 · Zeitliche Filter

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(14, 3.5), sharey=False)
signs["last_seen_at"].str[:7].value_counts().sort_index().plot(kind="bar", ax=achsen[0], title="alle Zeichen")

signs = cw.filter_stable_signs(signs, zuletzt_gesehen_nach, mindest_standzeit_monate)

signs["last_seen_at"].str[:7].value_counts().sort_index().plot(
    kind="bar", ax=achsen[1], title=f"zuletzt gesehen nach {zuletzt_gesehen_nach}, ≥ {mindest_standzeit_monate} Monate"
)
plt.tight_layout()

## 3 · Abstand zur OSM-Radinfrastruktur

Statt zweier Pufferläufe (25 m / 30 m) eine Abstandsspalte. Gepuffert wird nur um die
paar tausend Punkte; die 5,8 Mio. OSM-Linien werden nicht umprojiziert, sondern nur die
Treffer des Vorfilters metrisch nachgemessen.

In [ ]:
ways = gpd.read_parquet(pfad_radwege)
radinfra = cw.filter_cycle_infrastructure(ways)
print(f"Gesamtlänge: {cw.total_km(radinfra):,.2f} km".replace(",", "X").replace(".", ",").replace("X", "."))
del ways

In [ ]:
# Suchradius = größte Prioritätsschwelle; alles darüber interessiert nicht.
suchradius = max(schwelle for schwelle, _ in cw.PRIO_AB_DISTANZ)

signs["dist_cw_m"] = cw.distance_to_nearest(signs, radinfra, suchradius)
del radinfra

autobahnen = gpd.read_parquet(pfad_autobahnen)
signs["dist_mw_m"] = cw.distance_to_nearest(signs, autobahnen, cw.AUTOBAHN_ABSTAND_M)
del autobahnen

print(f"ohne Radinfra in {suchradius:.0f} m:  {(signs.dist_cw_m == float('inf')).sum():>6}")
print(f"Autobahn in {cw.AUTOBAHN_ABSTAND_M:.0f} m:      {(signs.dist_mw_m < float('inf')).sum():>6}")

## 4 · Kandidaten und Priorität

MapRoulette-Priorität: 0 = High, 1 = Medium, 2 = Low.
Schilder mit Radinfra näher als der kleinsten Schwelle fallen heraus, ebenso alles an
Autobahnen — Radwegzeichen an Auffahrten sind fast immer Fehlerkennungen.

In [ ]:
signs["prio"] = cw.assign_priority(signs["dist_cw_m"])

kandidaten = signs[signs["prio"].notna() & (signs["dist_mw_m"] == float("inf"))].reset_index(drop=True)
kandidaten["prio_text"] = kandidaten["prio"].map(cw.PRIO_TEXT)
kandidaten["VZ"] = kandidaten["value"].map(cw.RADWEG_ZEICHEN)

print(f"Kandidaten: {len(kandidaten)}")
kandidaten.groupby(["prio", "prio_text"]).size().to_frame("Anzahl")

In [ ]:
kandidaten.plot(column="prio", figsize=(6, 8), markersize=4, cmap="RdYlGn_r", legend=True)

## 5 · Bereits bestehende Aufgaben abziehen

In [ ]:
# Alles außer Fixed / Created / Skipped blockiert eine Neuanlage: was schon als
# "false positive" oder "already fixed" abgehakt wurde, soll nicht wiederkommen.
bestehende = cw.load_challenge_tasks(challenge_id, cw.load_token(pfad_config, "API_KEY_MAPROULETTE"))
print(f"blockierende Aufgaben in Challenge {challenge_id}: {len(bestehende)}")

kandidaten = cw.drop_near_existing_tasks(kandidaten, bestehende)
print(f"neu anzulegen: {len(kandidaten)}")

## 6 · Neuestes Mapillary-Bild

Der Kern der Überarbeitung. `1_` rief `mly.interface.feature_from_key` je Feature auf
und nahm `images[-1]` — das letzte Element einer Liste, die **nicht** nach Aufnahmezeit
sortiert ist. Hier kommt `captured_at` über die Feldexpansion der Graph-API gleich mit
(`fields=id,images{id,captured_at}`), und bis zu 50 Features gehen in einen Request.

In [ ]:
bilder = cw.newest_image_ids(kandidaten["id"], cw.load_token(pfad_config))

# Über den String-Index verbinden: Mapillary-IDs übersteigen 2**53 und würden
# als Zahl gerundet.
kandidaten = kandidaten.set_index(kandidaten["id"].astype(str)).join(bilder).reset_index(drop=True)

kandidaten[["id", "value", "last_seen_at", "dist_cw_m", "image_id", "image_captured_at"]].head()

### Plausibilitätsprüfung

`last_seen_at` aus dem Parquet ist die letzte Aufnahme, auf der das Zeichen erkannt
wurde. Wenn die Bildauswahl stimmt, muss das verlinkte Bild ungefähr von diesem Tag
sein. Beim Vorgänger lagen hier oft Jahre dazwischen.

In [ ]:
abstand_tage = (
    kandidaten["image_captured_at"].dt.tz_localize(None) - pd.to_datetime(kandidaten["last_seen_at"])
).dt.days.abs()

print(abstand_tage.describe().to_string())
print(f"\nBild am selben Tag wie last_seen_at: {(abstand_tage <= 1).sum()} von {abstand_tage.notna().sum()}")

abstand_tage.plot(kind="hist", bins=40, figsize=(8, 3), title="Tage zwischen verlinktem Bild und last_seen_at")

## 7 · GeoJSON für MapRoulette schreiben

Die Datei behält ihren Namen, damit MapRoulette denselben Eingabepfad behalten kann.
Deshalb wird der bisherige Inhalt **vor** dem Überschreiben eingelesen — sonst gäbe es
im nächsten Schritt nichts mehr zu vergleichen.

In [ ]:
vorheriger_stand = cw.read_geojson(ziel_geojson)

aufgaben = cw.build_maproulette_geojson(kandidaten)
cw.write_geojson(aufgaben, ziel_geojson)

print(aufgaben["features"][0]["properties"]["instruction"])

## 8 · Vergleich mit dem vorherigen Stand

Stellt die Aufgaben-IDs des bisherigen Datei-Inhalts denen dieses Laufs gegenüber. Der
interessante Wert ist **anderes Bild** — so viele Aufgaben verlinkten vorher ein anderes
(in aller Regel älteres) Foto.

Stammt der vorherige Stand aus `1_`, ist ein Mengenunterschied zu erwarten: die Datei
wurde zu einem anderen Zeitpunkt erzeugt, mit anderem Datenstand und einem anderen Stand
der Challenge. `git diff` auf die Datei zeigt dasselbe noch einmal zeilenweise.

In [ ]:
if vorheriger_stand:
    cw.print_comparison(cw.compare_task_sets(vorheriger_stand, aufgaben))
else:
    print("kein vorheriger Stand vorhanden - Vergleich übersprungen")

---

## Challenge-Beschreibung (für MapRoulette)

## 🚲 Fehlende Radwege anhand von Mapillary-Verkehrszeichen ergänzen (Deutschland)

Diese Challenge basiert auf automatisch erkannten, radverkehrsbezogenen Verkehrszeichen aus Mapillary-Bildern in Deutschland.

### 📌 Kriterien für jede Aufgabe

Nur Aufgaben, die **alle** folgenden Bedingungen erfüllen, wurden berücksichtigt:

- Das Verkehrszeichen wurde **in Mapillary erkannt**.
- Es handelt sich um ein **radverkehrsbezogenes Zeichen**
  *(z. B. gemeinsamer Geh- und Radweg, reiner Radweg, getrennter Geh-/Radweg)*.
- Das Zeichen wurde über **mindestens 9 Monate hinweg** gesehen.
- Die neueste Aufnahme stammt aus den letzten Monaten.
- Es existiert **kein OSM-"Radweg" innerhalb von 25 m** des Standortes.

### 🔍 Was du tun solltest

1. Öffne den Ort in **Mapillary** und **radinfra.de** sowie einem Editor.
2. Prüfe, ob an der Stelle eine **Radinfrastruktur fehlt**.
3. Falls ja, ergänze die passenden OSM-Tags:
   z. B. `highway=cycleway`, `cycleway=*`, `bicycle=designated`, etc.
4. Wenn bereits alles korrekt gemappt ist, kannst du die Aufgabe einfach **als erledigt markieren**.

🗺️ Vielen Dank für deine Hilfe beim Ausbau der Radinfrastruktur in OSM!

---

Task-Template für MapRoulette (Feld *Instruction*):

```
{{instruction}}
```